Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4/0001/0001_1_1_2_Augmented.png'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Preprocessing for Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]

# === Protocol 3 – 14 training samples per finger
PROTOCOL3_TRAIN_FILES = [
    ("1", ""), ("1", "1_Augmented"), ("1", "2_Augmented"), ("1", "3_Augmented"),
    ("2", ""), ("2", "1_Augmented"), ("2", "2_Augmented"), ("2", "3_Augmented"),
    ("3", "1_Augmented"), ("3", "2_Augmented"), ("3", "3_Augmented"),
    ("4", "1_Augmented"), ("4", "2_Augmented"), ("4", "3_Augmented")
]

train_data = []
train_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print(f"🔧 Applying denoising (h={h})...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === LOAD TRAINING DATA FOR STRATEGY 2, PROTOCOL 3 ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="🔄 Loading Protocol 3 - Strategy 2 Training"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_NUMS:
        for idx, (img_num, suffix) in enumerate(PROTOCOL3_TRAIN_FILES, start=1):
            if suffix == "":
                fname = f"{subj}_{finger}_{img_num}.png"
                label_suffix = "orig"
            else:
                fname = f"{subj}_{finger}_{img_num}_{suffix}.png"
                label_suffix = suffix

            img_path = os.path.join(subject_path, fname)
            print(f"\n📁 Subject {subj} - Finger {finger} - Sample {idx:02d} → {img_path}")

            if not os.path.exists(img_path):
                print(f"⚠️ Missing: {img_path}")
                continue

            # Step 1: Load & Resize
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            print(f"📏 Resized to {IMAGE_SIZE}")

            # Step 2: Denoising (apply before histogram equalization!)
            img_denoised = apply_denoising(img, h=10)
            print("🔧 Denoising applied.")

            # Step 3: Histogram Equalization
            img_eq = exposure.equalize_hist(img_denoised)  # float image in range [0,1]
            print("✨ Histogram equalization done.")

            # Step 4: Normalize
            img_eq_uint8 = (img_eq * 255).astype(np.uint8)  # Optional if needed for display/saving
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            print(f"📊 Normalized: mean={np.mean(img_norm):.2f}, std={np.std(img_norm):.2f}")
            # Step 5: Flatten & Save
            train_data.append(img_norm.flatten())
            label = os.path.splitext(fname)[0]  # 👈 keeps label same as file name
            train_labels.append(label)
            print(f"✅ Training sample saved as: {label}")

# === CONVERT TO NUMPY ARRAYS ===
train_data = np.array(train_data)
train_labels = np.array(train_labels)

# === SHAPE CHECK ===
print("\n📊 ✅ Final Train Data Shape:", train_data.shape)
print("📌 ✅ Train Labels Shape:", train_labels.shape)
print("🧾 ✅ First Few Labels:", train_labels[:5])


Test:

In [ ]:
import numpy as np

print("🔧 Starting PCA using Gram matrix...")

# === Step 1: Center the training data ===
print("📍 Centering training data...")
mean_vector = np.mean(train_data, axis=0)
centered_data = train_data - mean_vector  # Shape: (n_samples, n_features)

# === Step 2: Compute Gram matrix (subject-to-subject) ===
print("📐 Computing Gram matrix (size: subjects × subjects)...")
gram_matrix = centered_data @ centered_data.T  # Shape: (n_samples, n_samples)

# === Step 3: Eigen decomposition of Gram matrix ===
print("🧮 Performing eigen-decomposition of Gram matrix...")
eig_vals, eig_vecs = np.linalg.eigh(gram_matrix)  # Use eigh for symmetric matrix

# === Step 4: Sort eigenvalues and eigenvectors in descending order ===
print("📊 Sorting eigenvalues and eigenvectors...")
sorted_indices = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[sorted_indices]
eig_vecs = eig_vecs[:, sorted_indices]

# === Step 5: Filter valid eigenvalues (> 1e-10 for stability) ===
print("✅ Filtering out near-zero eigenvalues...")
valid_mask = eig_vals > 1e-10
eig_vals_valid = eig_vals[valid_mask]
eig_vecs_valid = eig_vecs[:, valid_mask]
print(f"ℹ️ Retained {len(eig_vals_valid)} valid eigenvalues.")

# === Step 6: Project eigenvectors back to original feature space ===
print("↩️ Mapping eigenvectors to original feature space...")
eig_vecs_full = (centered_data.T @ eig_vecs_valid) / np.sqrt(eig_vals_valid)

# === Step 6.5: Normalize eigenvectors (optional but recommended) ===
print("🔄 Normalizing eigenvectors...")
eig_vecs_full = eig_vecs_full / np.linalg.norm(eig_vecs_full, axis=0)

# === Step 7: Project training data into PCA space ===
print("📤 Projecting centered data to PCA space...")
train_data_pca = centered_data @ eig_vecs_full  # Shape: (n_samples, n_components)

# === Final Outputs ===
print("\n✅ PCA projection completed successfully.")
print("📐 Transformed training data shape:", train_data_pca.shape)
print("📊 Number of principal components used:", eig_vecs_full.shape[1])


Preprocessing for Testing:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = "/content/drive/MyDrive/Datasets/Final UTFVP_Original with Augmented Images_4"
IMAGE_SIZE = (128, 60)
FINGER_NUMS = [1, 2, 3, 4, 5, 6]

# Protocol 3 test files (originals only)
PROTOCOL3_TEST_FILES = ["3", "4"]  # image3 and image4 originals

test_data = []
test_labels = []

# === FUNCTION: Denoising ===
def apply_denoising(image, h=10):
    print(f"🔧 Applying denoising (h={h})...")
    return cv2.fastNlMeansDenoising(image, h=h)

# === LOAD TEST DATA FOR STRATEGY 2, PROTOCOL 3 ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="🧪 Loading Protocol 3 - Strategy 2 Test"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    for finger in FINGER_NUMS:
        for img_num in PROTOCOL3_TEST_FILES:
            fname = f"{subj}_{finger}_{img_num}.png"
            img_path = os.path.join(subject_path, fname)
            print(f"\n📁 Subject {subj} - Finger {finger} - Test Image {img_num}")
            print(f"🖼️ Loading: {img_path}")

            if not os.path.exists(img_path):
                print(f"⚠️ Missing: {img_path}")
                continue

            # Step 1: Load & Resize
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            img = cv2.resize(img, IMAGE_SIZE)
            print(f"📏 Resized to {IMAGE_SIZE}")

            # Step 2: Denoising (apply before histogram equalization!)
            img_denoised = apply_denoising(img, h=10)
            print("🔧 Denoising applied.")

            # Step 3: Histogram Equalization
            img_eq = exposure.equalize_hist(img_denoised)  # float image in range [0,1]
            print("✨ Histogram equalization done.")

            # Step 4: Normalize
            img_eq_uint8 = (img_eq * 255).astype(np.uint8)  # Optional if needed for display/saving
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            print(f"📊 Normalized: mean={np.mean(img_norm):.2f}, std={np.std(img_norm):.2f}")

            # Step 5: Flatten & Save
            test_data.append(img_norm.flatten())
            label = os.path.splitext(fname)[0]  # 👈 keeps label same as file name
            test_labels.append(label)
            print(f"✅ Test sample saved as: {label}")

# === CONVERT TO NUMPY ARRAYS ===
test_data = np.array(test_data)
test_labels = np.array(test_labels)

# === SHAPE CHECK ===
print("\n📊 ✅ Final Test Data Shape:", test_data.shape)
print("📌 ✅ Test Labels Shape:", test_labels.shape)
print("🧾 ✅ First Few Test Labels:", test_labels[:5])


Benchmarking

In [ ]:
correct_matches = 0
total_tests = len(test_data)

# === Step 1: Project test data into PCA space
centered_test_data = test_data - mean_vector
proj_test_data = centered_test_data @ eig_vecs_full
print("\n📤 Test data projected into PCA space.")

# === Step 2: Match each test sample against training set
for i in range(total_tests):
    proj_test = proj_test_data[i]
    true_label = test_labels[i]  # e.g., "0032_finger3_test_4_orig"

    # 📏 Manhattan distance to all training vectors
    distances = np.sum(np.abs(train_data_pca - proj_test), axis=1)

    # 🏆 Find the closest match
    closest_index = np.argmin(distances)
    predicted_label = train_labels[closest_index]  # e.g., "0032_finger3_aug05"

    # 🎯 Extract subject ID and finger number
    true_parts = true_label.split('_')
    pred_parts = predicted_label.split('_')

    true_subject = true_parts[0]
    true_finger = true_parts[1]  # e.g., "finger3"

    pred_subject = pred_parts[0]
    pred_finger = pred_parts[1]  # e.g., "finger3"

    # ✅ Match check: subject AND finger
    if pred_subject == true_subject and pred_finger == true_finger:
        match_result = "✅"
        correct_matches += 1
    else:
        match_result = "❌"

    print(f"🔍 Test {i+1:03d}: Predicted = {predicted_label}, Actual = {true_label} {match_result}")

# === Final Accuracy Report ===
accuracy = (correct_matches / total_tests) * 100
print(f"\n🎉 Finger-wise Identification Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests} correct matches)")
